# MedGuide | PaddleOCR وDINOv2 والتقييم
فكّي حزمة المشروع في `MyDrive/MedGuide2`. شغّلي خلايا تنزيل MedPackID في دفتر 03 أولًا. تنزيل الصور يتم داخل Colab إلى Drive.
تستخرج الخلايا التالية قصاصات العلب من مربعات الناشر. لإضافة مرجع بصري إلى التطبيق، استخدمي جدولًا يربط كل قصاصة بسجل صحيح في قاعدة MedGuide.
توجد قوالب شواهد البيانات في `notebooks/templates/`، وشرح حقولها في `docs/RUN_VISION.md`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, sys

drive.mount('/content/drive')
project = Path('/content/drive/MyDrive/MedGuide2')
data = Path('/content/drive/MyDrive/MedGuideData')
archives = data / 'MedPackID_v1'
run_dir = data / 'vision_artifacts'
if not (project / 'engine/pyproject.toml').is_file():
    raise FileNotFoundError(project)
os.chdir(project)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', './engine',
                       '-r', 'engine/requirements-vision.txt'])

## قراءة النص من قصاصات مربعات الناشر
ينشئ التقرير قراءة عربية ولاتينية منفصلة. إذا جهّزتِ تفريغًا يدويًا، اجعلي المجلد يحتوي القصاصات نفسها التي وصفها الملف `ocr_ground_truth.csv`. التجربة دون ملف التفريغ تحفظ النتائج وتترك CER وWER فارغين.

In [ ]:
def invoke(module, *args):
    subprocess.check_call([sys.executable, '-m', module, *map(str, args)], cwd=project)

paddle_root = data / 'PP_OCRv5_Mobile'
invoke('engine.eval.fetch_ocr_models', paddle_root)
crops = run_dir / 'crops'
invoke('engine.eval.medpack_crops', archives, crops, '--limit', 30)
ocr_reference = data / 'ocr_ground_truth.csv'
invoke('engine.eval.workflows', 'ocr', '--images', crops, '--output', run_dir / 'paddleocr.json',
       '--model-root', paddle_root,
       *(['--manifest', ocr_reference] if ocr_reference.exists() else []))

## الاسترجاع المرتبط بمنتجات القاعدة
ملف `visual_reference.csv` يصف قصاصات المجلد `reference_images`. ملف `visual_queries.csv` يصف صورًا أخرى مستقلة في `retrieval_test`. يجب وجود صور مرجعية للمنتج المقاس وصور اختبار من لقطات مستقلة.

In [ ]:
reference = data / 'visual_reference.csv'
gallery = run_dir / 'index'
if reference.is_file():
    invoke('engine.eval.workflows', 'build-visual', '--images', data / 'reference_images',
           '--manifest', reference, '--output', gallery)
    queries = data / 'visual_queries.csv'
    if queries.is_file():
        invoke('engine.eval.workflows', 'retrieval', '--index', gallery, '--images', data / 'retrieval_test',
               '--manifest', queries, '--output', run_dir / 'retrieval.json')
    else:
        print('Recall@1 / Recall@5 / MRR: يلزم visual_queries.csv وصور الاسترجاع المستقلة.')
else:
    print('يلزم visual_reference.csv وصور العلب المرتبطة بقاعدة MedGuide.')

## التقييم المستقل
اضبطي `DETECTOR_RUN` على مجلد `product_split_...` الذي يحتوي `fit/weights/best.pt` و`dataset_manifest.json`.
دون أحد مكونات المسار يقيس التقرير المكونات المفعّلة ويصرّح بذلك. لا يصلح هذا التقرير الجزئي للحكم على اكتمال التطبيق.

In [ ]:
DETECTOR_RUN = None  # ضعي مسار مجلد product_split_... بعد التدريب
test = data / 'recognition_test.csv'
options = ['--ocr', '--model-root', paddle_root]
if (gallery / 'visual_manifest.json').is_file():
    options += ['--index', gallery]
if DETECTOR_RUN:
    detector_run = Path(DETECTOR_RUN)
    options += ['--detector', detector_run / 'fit/weights/best.pt',
                '--detector-manifest', detector_run / 'dataset_manifest.json']
if test.is_file():
    invoke('engine.eval.workflows', 'full', '--images', data / 'recognition_test', '--manifest', test,
           '--output', run_dir / 'recognition_test.json', *options)
else:
    print('يلزم recognition_test.csv ومجلد صور اختبار مستقل لحساب التعرف الخاطئ وزمن المسار.')

## إنتاج ONNX
يستخدم التصدير ملفات Paddle الرسمية التي حفظت الخلايا السابقة بصمتها. يختبر السكربت مخرجات ONNX محليًا ويكتب تقارير التوافق العددي.

In [ ]:
exports = run_dir / 'mobile'
invoke('engine.eval.export_models', 'ocr', '--source', paddle_root, '--output', exports)
invoke('engine.eval.export_models', 'dino', '--source', crops, '--output', exports)